# Capacity Scenario Analysis: Integer Slot Increments

Evaluates the impact of adding **+1 through +4 integer slots** to each provider/diagnostic category.
The fractional-multiplier approach (105–120%) collapses to the same integer slot count within each group
due to the simulation's `> 0` / `−= 1` slot-tracking logic, so this notebook tests true integer steps.

| Scenario Group | Baseline | Steps tested |
|---|---|---|
| **GYN Provider** | 4 slots/day (cytology · HPV-alone · co-test) | 5 · 6 · 7 · 8 |
| **PCP Throughput** | 2 patients/day (sim scale) | 3 · 4 · 5 · 6 |
| **LDCT** | 4 slots/day | 5 · 6 · 7 · 8 |
| **Cervical Biopsy** | 4 slots/day (colposcopy) | 5 · 6 · 7 · 8 |

Each scenario runs **N_SEEDS** independent 80-year Monte Carlo simulations.  
Box plots show IQR with **5th–95th percentile** whiskers across seeds.

In [ ]:
import sys
import re
import time
import colorsys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
})

# ── path setup ────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent.parent
for _sub in ['src', 'ModelParameters']:
    _p = str(PROJECT_ROOT / _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)

import parameters as cfg
from scenarios import ScenarioConfig
from mc_scenarios import run_mc_scenario

print(f'Project root : {PROJECT_ROOT}')
print(f'DAILY_PATIENTS: {cfg.DAILY_PATIENTS}')
print(f'CAPACITIES   : {cfg.CAPACITIES}')

In [ ]:
# ── simulation settings ───────────────────────────────────────────────────
N_SEEDS      = 50      # seeds per scenario
N_WORKERS    = 3       # capped to avoid memory pressure on 8 GB RAM
SEED_START   = 42
WARMUP_YEARS = cfg.WARMUP_YEARS   # 10

# integer slot steps to add above baseline
SLOT_STEPS = [1, 2, 3, 4]

# output paths
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'Scenario Analysis Visualizations' / 'Capacity Scenario Integer'
DATA_DIR   = PROJECT_ROOT / 'src' / 'mc_scenario_data' / 'capacity_scenarios_integer'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# snapshot baseline values
_BASE_CAP = dict(cfg.CAPACITIES)   # e.g. cytology=4, ldct=4, …
_BASE_DP  = cfg.DAILY_PATIENTS     # 2

print(f'N_SEEDS    : {N_SEEDS}')
print(f'N_WORKERS  : {N_WORKERS}')
print(f'WARMUP_YEARS: {WARMUP_YEARS}')
print(f'SLOT_STEPS : {SLOT_STEPS}')

In [ ]:
# ── scenario group definitions ────────────────────────────────────────────
SCENARIO_GROUPS = {
    'GYN': {
        'label':    'GYN Provider Capacity',
        'subtitle': 'Primary cervical screening slots (cytology · HPV-alone · co-test)',
        'color':    '#C2185B',
        'params':   {
            'CAPACITIES.cytology':  _BASE_CAP['cytology'],
            'CAPACITIES.hpv_alone': _BASE_CAP['hpv_alone'],
            'CAPACITIES.co_test':   _BASE_CAP['co_test'],
        },
    },
    'PCP': {
        'label':    'PCP Throughput',
        'subtitle': 'Daily provider visit capacity — all outpatient providers (PCP + GYN)',
        'color':    '#1565C0',
        'params':   {
            'DAILY_PATIENTS': _BASE_DP,
        },
    },
    'LDCT': {
        'label':    'LDCT Capacity',
        'subtitle': 'Daily low-dose CT slots for lung cancer screening',
        'color':    '#E65100',
        'params':   {
            'CAPACITIES.ldct': _BASE_CAP['ldct'],
        },
    },
    'Biopsy': {
        'label':    'Cervical Biopsy Capacity',
        'subtitle': 'Daily colposcopy slots for cervical cancer follow-up diagnostics',
        'color':    '#2E7D32',
        'params':   {
            'CAPACITIES.colposcopy': _BASE_CAP['colposcopy'],
        },
    },
}

print('Scenario groups and absolute slot values tested:')
for gk, g in SCENARIO_GROUPS.items():
    print(f'\n{gk} — {g["label"]}')
    for param, base in g['params'].items():
        vals = '  /  '.join(str(int(base + s)) for s in SLOT_STEPS)
        print(f'  {param}: baseline={int(base)}  →  {vals}')

In [ ]:
# ── build ScenarioConfig objects (4 groups × 4 steps = 16 scenarios) ──────
scenarios = {}   # name → (ScenarioConfig, group_key, step)

for group_key, group in SCENARIO_GROUPS.items():
    for step in SLOT_STEPS:
        name = f'{group_key.lower()}_plus{step}'
        overrides = {
            param: int(base_val + step)
            for param, base_val in group['params'].items()
        }
        sc = ScenarioConfig(
            name=name,
            description=f'{group["label"]} +{step} slot(s) above baseline',
            param_overrides=overrides,
            cotesting_enabled=False,
        )
        scenarios[name] = (sc, group_key, step)

print(f'Total scenarios: {len(scenarios)}')
for name, (sc, gk, s) in scenarios.items():
    print(f'  {name:22s}  overrides = {sc.param_overrides}')

## Run / Load Simulations

Results are **cached** to CSV. Re-running this cell skips completed scenarios.

⏱ With `N_WORKERS = 3`: ~15–20 min per new scenario

In [ ]:
csv_paths = {}   # name → Path

for name, (sc, group_key, step) in scenarios.items():
    csv_path = DATA_DIR / f'{name}_n{N_SEEDS}_start{SEED_START}.csv'
    if csv_path.exists():
        print(f'[CACHED]  {name}')
    else:
        t0 = time.time()
        print(f'[RUNNING] {name} …', end=' ', flush=True)
        run_mc_scenario(
            sc,
            n_seeds=N_SEEDS,
            seed_start=SEED_START,
            n_workers=N_WORKERS,
            out_csv=str(csv_path),
            progress=False,
        )
        elapsed = (time.time() - t0) / 60
        print(f'done ({elapsed:.1f} min)')
    csv_paths[name] = csv_path

print('\nAll scenarios ready.')

In [ ]:
# ── locate baseline CSV (highest n available) ─────────────────────────────
def _parse_n(p: Path) -> int:
    m = re.search(r'_n(\d+)_', p.stem)
    return int(m.group(1)) if m else 0

_bl_dir = PROJECT_ROOT / 'src' / 'mc_baseline_data'
_candidates = sorted(_bl_dir.glob('*baseline*_start42.csv'), key=_parse_n)
if not _candidates:
    _candidates = sorted(_bl_dir.glob('*.csv'), key=_parse_n)
if not _candidates:
    raise FileNotFoundError(
        f'No baseline CSV found in {_bl_dir}. '
        'Run the baseline Monte Carlo simulation first.'
    )
BASELINE_CSV = _candidates[-1]
print(f'Using baseline: {BASELINE_CSV.name}  ({_parse_n(BASELINE_CSV)} seeds)')

In [ ]:
# ── metrics to display ────────────────────────────────────────────────────
PLOT_METRICS = [
    ('revenue_capture_rate_pct',    'Revenue Capture Rate',      '%'),
    ('population_capture_rate_pct', 'Population Capture Rate',   '%'),
    ('mean_wait_primary_days',      'Primary Screening Wait',    'days'),
    ('mean_wait_secondary_days',    'Secondary Procedure Wait',  'days'),
    ('ltfu_rate_primary_pct',       'Primary LTFU Rate',         '%'),
    ('ltfu_rate_secondary_pct',     'Secondary LTFU Rate',       '%'),
]


def load_per_seed(csv_path: Path, warmup: int = 10) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    records = []
    for seed, sdf in df.groupby('seed'):
        row = {'seed': seed}
        for metric_key, *_ in PLOT_METRICS:
            sub = sdf[sdf['metric'] == metric_key]
            if sub.empty:
                row[metric_key] = np.nan
                continue
            if sub['year'].nunique() > 1:
                sub = sub[sub['year'] >= warmup]
            row[metric_key] = sub['value'].mean()
        records.append(row)
    return pd.DataFrame(records).set_index('seed')


print('Loading baseline …', end=' ', flush=True)
baseline_df = load_per_seed(BASELINE_CSV, warmup=WARMUP_YEARS)
print(f'{len(baseline_df)} seeds loaded')

scenario_dfs = {}
for name, csv_path in csv_paths.items():
    scenario_dfs[name] = load_per_seed(csv_path, warmup=WARMUP_YEARS)

print(f'Scenario datasets loaded: {len(scenario_dfs)}')
baseline_df.describe().round(2)

In [ ]:
# ── visualization helpers ─────────────────────────────────────────────────

def shade_palette(hex_color: str, n: int = 4,
                  lo_lightness: float = 0.80,
                  hi_lightness: float = None) -> list:
    r, g, b = mcolors.to_rgb(hex_color)
    h, nat_l, s = colorsys.rgb_to_hls(r, g, b)
    hi = hi_lightness if hi_lightness is not None else max(nat_l, 0.25)
    lightnesses = np.linspace(lo_lightness, hi, n)
    return [colorsys.hls_to_rgb(h, float(lt), s) for lt in lightnesses]


def apply_y_format(ax, unit: str) -> None:
    if unit == '%':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: f'{x:.1f}%'))
    elif unit == 'days':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(
            lambda x, _: f'{x:.0f}'))


def draw_boxes(ax, data_dict: dict, colors: list,
               highlight_baseline: bool = True) -> None:
    labels = list(data_dict.keys())
    data   = [np.asarray(v, dtype=float) for v in data_dict.values()]
    data   = [v[~np.isnan(v)] for v in data]

    bp = ax.boxplot(
        data,
        labels=labels,
        whis=1.5,            # standard IQR whiskers (cleaner for expo)
        showfliers=False,
        patch_artist=True,
        widths=0.52,
        medianprops=dict(color='white', linewidth=3.0),   # white pops on colored boxes
        boxprops=dict(linewidth=1.3),
        whiskerprops=dict(linewidth=1.3, linestyle='--'),
        capprops=dict(linewidth=1.6),
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.88)

    if highlight_baseline and len(data[0]) > 0:
        ax.axhline(np.median(data[0]),
                   color='#546E7A', linewidth=1.2,
                   linestyle=':', alpha=0.7)


GRAY_BASE = '#90A4AE'

print('Helpers defined.')

## Per-Group Box Plot Figures

One figure per scenario group (4 total), each with 6 metric subplots.  
Box = IQR · whiskers = 5th–95th pct across seeds · dotted line = baseline median.  
X-axis labels show **absolute slot counts** (not multipliers).

In [ ]:
LTFU_ANNOTATION = (
    "↑ More intake patients compete for\n"
    "fixed procedure slots → longer queues\n"
    "→ higher LTFU"
)

for group_key, group in SCENARIO_GROUPS.items():
    shades     = shade_palette(group['color'], n=len(SLOT_STEPS))
    all_colors = [GRAY_BASE] + shades

    _base_val = next(iter(group['params'].values()))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes_flat = axes.flatten()

    for ax_i, (metric_key, metric_label, unit) in enumerate(PLOT_METRICS):
        ax = axes_flat[ax_i]

        data_dict = {}
        data_dict['Baseline\n(%i)' % int(_base_val)] = (
            baseline_df[metric_key].values
            if metric_key in baseline_df.columns else np.array([])
        )

        for step in SLOT_STEPS:
            name = f'{group_key.lower()}_plus{step}'
            sdf  = scenario_dfs.get(name)
            arr  = sdf[metric_key].values if (sdf is not None and metric_key in sdf.columns) else np.array([])
            label = '+%i\n(%i)' % (step, int(_base_val + step))
            data_dict[label] = arr

        draw_boxes(ax, data_dict, all_colors)
        apply_y_format(ax, unit)

        ax.set_title(metric_label, fontweight='bold', fontsize=14, pad=8)
        ax.set_xlabel('Slots added  (absolute)', fontsize=12, labelpad=4)
        if unit in ('%', 'days'):
            ax.set_ylabel(unit, fontsize=12)
        ax.grid(True, alpha=0.22, axis='y', linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='x', labelsize=11)
        ax.tick_params(axis='y', labelsize=11)

        # annotate the PCP primary-LTFU tradeoff
        if group_key == 'PCP' and metric_key == 'ltfu_rate_primary_pct':
            ax.annotate(
                LTFU_ANNOTATION,
                xy=(0.97, 0.97), xycoords='axes fraction',
                ha='right', va='top', fontsize=9,
                color='#B71C1C',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFEBEE',
                          edgecolor='#EF9A9A', alpha=0.9),
            )

    legend_handles = [
        Patch(facecolor=GRAY_BASE, label='Baseline (%i slots)' % int(_base_val), alpha=0.88)
    ] + [
        Patch(facecolor=shades[j],
              label='+%i slot%s (%i total)' % (SLOT_STEPS[j], 's' if SLOT_STEPS[j] > 1 else '', int(_base_val + SLOT_STEPS[j])),
              alpha=0.88)
        for j in range(len(SLOT_STEPS))
    ]
    fig.legend(
        handles=legend_handles,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.03),
        ncol=5,
        framealpha=0.95,
        fontsize=12,
    )

    fig.suptitle(
        f'Capacity Scenario Analysis: {group["label"]}\n'
        f'{group["subtitle"]}',
        fontsize=15, fontweight='bold', y=1.01,
    )
    plt.tight_layout(rect=[0, 0.04, 1, 1])

    out_path = OUTPUT_DIR / f'01_boxplot_{group_key.lower()}_integer.png'
    fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'Saved → {out_path.name}')
    plt.show()

## Summary: Median % Change from Baseline

Line chart comparing **median % change** from baseline across all four scenario groups for each metric.
X-axis = integer slots added above baseline.

In [ ]:
baseline_medians = {
    mk: baseline_df[mk].dropna().median()
    for mk, *_ in PLOT_METRICS
}

summary_rows = []
for group_key, group in SCENARIO_GROUPS.items():
    for step in SLOT_STEPS:
        name = f'{group_key.lower()}_plus{step}'
        sdf  = scenario_dfs.get(name)
        row  = {'Group': group['label'], 'GroupKey': group_key, 'Step': step}
        for mk, ml, _ in PLOT_METRICS:
            b_med = baseline_medians.get(mk, 0)
            if sdf is not None and mk in sdf.columns and b_med != 0:
                row[mk] = 100.0 * (sdf[mk].dropna().median() - b_med) / abs(b_med)
            else:
                row[mk] = np.nan
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes_flat = axes.flatten()

for ax_i, (metric_key, metric_label, unit) in enumerate(PLOT_METRICS):
    ax = axes_flat[ax_i]

    for group_key, group in SCENARIO_GROUPS.items():
        sub    = summary_df[summary_df['GroupKey'] == group_key]
        y_vals = sub.set_index('Step')[metric_key].reindex(SLOT_STEPS).values

        ax.plot(
            SLOT_STEPS, y_vals,
            marker='o', linewidth=2.5, markersize=9,
            color=group['color'], label=group['label'],
            zorder=3,
        )

    ax.axhline(0, color='#455A64', linewidth=1.1, linestyle='--', alpha=0.55)
    ax.set_title(metric_label, fontweight='bold', fontsize=14, pad=8)
    ax.set_xlabel('Integer slots added above baseline', fontsize=12, labelpad=4)
    ax.set_ylabel('Median Δ from baseline (%)', fontsize=12)
    ax.set_xticks(SLOT_STEPS)
    ax.set_xticklabels([f'+{s}' for s in SLOT_STEPS], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(True, alpha=0.22, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

legend_handles = [
    plt.Line2D([0], [0],
               color=g['color'], marker='o',
               linewidth=2.5, markersize=9,
               label=g['label'])
    for g in SCENARIO_GROUPS.values()
]
fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.03),
    ncol=4,
    fontsize=12,
    framealpha=0.95,
)
fig.suptitle(
    'Capacity Scenario Summary: Median % Change from Baseline\n'
    'by scenario group and integer slot increment',
    fontsize=15, fontweight='bold', y=1.01,
)
plt.tight_layout(rect=[0, 0.04, 1, 1])

out_path = OUTPUT_DIR / '05_summary_pct_change_integer.png'
fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved → {out_path.name}')
plt.show()

In [ ]:
# ── tabular summary at +4 slots (largest increment) ───────────────────────
peak = summary_df[summary_df['Step'] == 4].set_index('Group')
cols = [mk for mk, *_ in PLOT_METRICS]
display_peak = peak[cols].copy()
display_peak.columns = [ml for _, ml, _ in PLOT_METRICS]
display_peak = display_peak.round(2)
print('Median % change from baseline at +4 slots:')
display_peak